# Zorse: Example on Chameleon Cloud

This notebook is a **Trovi artifact** that demonstrates how to run [Zorse](https://github.com/benson-guo/zorse) on Chameleon Cloud infrastructure.

**Originally created and tested on a 4x NVIDIA Tesla V100-PCIE-32GB bare-metal node.**

## Prerequisites

Before running this notebook, you must have:

1. An active **Chameleon Cloud** account
2. An existing **lease** on a GPU node (V100 or similar) at **CHI@UC**, **CHI@TACC**, or any other Chameleon site
3. The lease should reserve a bare-metal node with NVIDIA GPUs
4. This notebook should be run from the Chameleon JupyterHub environment

## Overview

The notebook walks through the following steps:

1. **Configure** training parameters and infrastructure settings
2. **Provision** a bare-metal GPU server on Chameleon
3. **Install** dependencies (CUDA toolkit, PyTorch, Zorse)
4. **Profile** model performance on the available GPUs
5. **Plan** an optimal distributed training configuration via the Zorse planner
6. **Train** using Zorse's heterogeneity-aware distributed training

## Step 1: Configuration

Set your lease name, server name, and training hyperparameters below. Update `lease_name` and `server_name` to match your Chameleon lease.

In [ ]:
lease_name = "your-lease-name"
server_name = "your-server-name"
image_name = "CC-Ubuntu24.04-CUDA"

model_name = "deepspeedllamav2_tiny"
sequence_length = 512
vocab_size = 49152
global_batch_size = 128
batch_size = 4
dtype = "float16"
master_port = 12347
warmup_iterations = 4
iterations = 4

num_gpus = 4

CONDA = "source $HOME/miniconda3/etc/profile.d/conda.sh && conda activate zorse"

## Step 2: Connect to Chameleon

Select your Chameleon project and site. The lease must already exist at the chosen site.

In [ ]:
import chi
from chi import context, lease, server

context.version = "1.0"
context.choose_project()
context.choose_site(default="CHI@UC")

## Step 3: Verify Lease and Launch Server

Verify that your lease is active, then launch a bare-metal server from it.

In [ ]:
l = lease.get_lease(lease_name)
l.show()

In [ ]:
s = server.Server(
    server_name,
    reservation_id=lease.get_node_reservation(lease_name),
    image_name=image_name,
)
s.wait()
s.associate_floating_ip()
s.refresh()
s.check_connectivity()

## Step 4: Verify GPU Access

Confirm that the GPUs are visible on the server.

In [ ]:
floating_ip = s.get_floating_ip()
print(f"Floating IP: {floating_ip}")
s.execute("nvidia-smi")

## Step 5: Install Dependencies

Install system packages, Miniconda, PyTorch, and Zorse dependencies. This may take several minutes.

In [ ]:
s.execute(
    "sudo apt-get update && sudo apt-get install -y nvtop git && "
    "wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh && "
    "bash /tmp/miniconda.sh -b -p $HOME/miniconda3 && "
    "rm /tmp/miniconda.sh && "
    "$HOME/miniconda3/bin/conda init bash && "
    "source $HOME/miniconda3/etc/profile.d/conda.sh && "
    "conda create -n zorse python=3.12 -y && "
    "conda activate zorse && "
    "pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121 && "
    "pip install transformers==4.46.2 matplotlib timer kneed scikit-learn pulp pymetis"
)

## Step 6: Clone Zorse and Verify Installation

Clone the Zorse repository and verify that PyTorch can see all GPUs.

In [ ]:
s.execute("git clone https://github.com/benson-guo/zorse.git ~/zorse")
s.execute(
    f"{CONDA} && python3 -c \"import torch; "
    "print('PyTorch:', torch.__version__); "
    "print('CUDA:', torch.cuda.is_available()); "
    "print('GPUs:', torch.cuda.device_count()); "
    "[print(f'  [{i}] {torch.cuda.get_device_name(i)}') for i in range(torch.cuda.device_count())]\""
)

## Step 7: Setup SSH for Distributed Training

Configure passwordless SSH (needed for multi-process distributed training) and detect the network interface.

In [ ]:
s.execute(
    "rm -f ~/.ssh/id_ed25519 ~/.ssh/id_ed25519.pub && "
    'ssh-keygen -t ed25519 -f ~/.ssh/id_ed25519 -N "" -q && '
    "cat ~/.ssh/id_ed25519.pub >> ~/.ssh/authorized_keys && "
    "chmod 600 ~/.ssh/authorized_keys && "
    "ssh-keyscan -H localhost >> ~/.ssh/known_hosts 2>/dev/null && "
    f"ssh-keyscan -H {floating_ip} >> ~/.ssh/known_hosts 2>/dev/null && "
    "ssh localhost hostname"
)

In [ ]:
ifname = s.execute(
    "ip route get 8.8.8.8 | head -1 | awk '{print $5}'"
).stdout.strip()
print(f"Network interface: {ifname}")

## Step 8: Profile Model on GPUs

Profile the model's compute and memory characteristics on the available GPUs. This information is used by the Zorse planner to determine the optimal parallelism strategy.

In [ ]:
print(f"Profiling {model_name} on V100...")
s.execute(
    f"{CONDA} && cd ~/zorse && ./profile_models.sh {dtype} {sequence_length} {master_port} {model_name}"
)

## Step 9: Create Hostfile and Collect Cluster Info

Create a hostfile describing the cluster topology, then collect GPU information and inter-GPU bandwidth measurements.

In [ ]:
hostfile_content = (
    f"{floating_ip},,/home/cc/miniconda3/etc/profile.d/conda.sh,zorse,/home/cc/zorse,{ifname}"
)

s.execute(f'cat > ~/zorse/hostfile << "EOF"\n{hostfile_content}\nEOF')
print("Hostfile:")
s.execute("cat ~/zorse/hostfile")

In [ ]:
s.execute(f"sudo ip addr add {floating_ip}/32 dev lo || true")
print("Floating IP added to loopback")

print("Collecting Cluster Info")
s.execute(
    f"{CONDA} && cd ~/zorse && python3 collect_cluster_info.py "
    "--machine_file hostfile "
    "--ib_disable "
    "--output cluster_info.json"
)

In [ ]:
s.execute("cat ~/zorse/cluster_info.json")

## Step 10: Run the Zorse Planner

The Zorse planner analyzes the cluster topology and model profile to determine the optimal distributed training configuration (pipeline parallelism, tensor parallelism, data parallelism, etc.).

In [ ]:
s.execute("cd ~/zorse && git pull")

In [ ]:
s.execute(
    f"{CONDA} && cd ~/zorse && python3 zorse_planner.py "
    f"--cluster_info_file cluster_info.json "
    f"--machine_file hostfile "
    f"--model_name {model_name} "
    f"--global_batch_size {global_batch_size} "
    f"--sequence_length {sequence_length} "
    "--nccl_ib_disable "
    "--use_agrs_comm_model "
    "--output_file training_config.json"
)

In [ ]:
print("Planner output:")
s.execute("cat ~/zorse/training_config.json")

## Step 11: Run Zorse Training

Launch distributed training using the Zorse-generated configuration. This uses the optimal parallelism strategy determined by the planner.

In [ ]:
zorse_cmd = (
    f"{CONDA} && cd ~/zorse && "
    f"NCCL_IB_DISABLE=1 "
    f"NCCL_SOCKET_IFNAME={ifname} "
    f"GLOO_SOCKET_IFNAME={ifname} "
    f"torchrun "
    f"--nproc_per_node={num_gpus} "
    f"--nnodes=1 "
    f"--node_rank=0 "
    f"--master_addr=localhost "
    f"--master_port={master_port} "
    f"zorse.py "
    f"--config_file training_config.json "
    f"--zero2_pipeline "
    f"--gloo_p2p "
    f"--offload_model_params "
    f"--optimizer_in_backwards"
)

print("Running Zorse...")
s.execute(f"bash -c '{zorse_cmd}' 2>&1 | tee ~/zorse/zorse.log")

## Cleanup

Delete the server when finished to release the resources.

In [ ]:
s.delete()